In [1]:
import subprocess
subprocess.run(['pip', 'install', 'psycopg2-binary', 'sqlalchemy'])

  Using cached psycopg2_binary-2.9.12-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (4.9 kB)
  Using cached sqlalchemy-2.0.49-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (9.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 430.6 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 425.4 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.4/611.4 kB 271.4 kB/s eta 0:00:0000:0100:01


CompletedProcess(args=['pip', 'install', 'psycopg2-binary', 'sqlalchemy'], returncode=0)

In [7]:
conn.rollback()
print("Transaction rolled back! ✅")

Transaction rolled back! ✅


In [15]:
cursor.close()
conn.close()

conn = psycopg2.connect(
    host="localhost",
    database="bank_reviews",
    user="yordanos",
    password="123456"
)
cursor = conn.cursor()
print("Reconnected! ✅")

Reconnected! ✅


In [16]:
conn.rollback()
cursor.execute("GRANT ALL ON SCHEMA public TO yordanos;")
conn.commit()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS banks (
        bank_id SERIAL PRIMARY KEY,
        bank_name VARCHAR(100),
        app_name VARCHAR(100)
    );
""")
conn.commit()
print("Banks table created! ✅")

Banks table created! ✅


In [17]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS reviews (
        review_id SERIAL PRIMARY KEY,
        bank_id INTEGER REFERENCES banks(bank_id),
        review_text TEXT,
        rating INTEGER,
        review_date DATE,
        sentiment_label VARCHAR(20),
        sentiment_score FLOAT,
        identified_theme VARCHAR(50),
        source VARCHAR(50)
    );
""")
conn.commit()
print("Reviews table created! ✅")

Reviews table created! ✅


In [19]:
import pandas as pd

df = pd.read_csv('data/raw/reviews.csv')

banks = df['bank'].unique()
for bank in banks:
    cursor.execute(
        "INSERT INTO banks (bank_name, app_name) VALUES (%s, %s) ON CONFLICT DO NOTHING;",
        (bank, bank)
    )
conn.commit()
print("Banks inserted! ✅")

# Check
cursor.execute("SELECT * FROM banks;")
print(cursor.fetchall())

Banks inserted! ✅
[(1, 'Commercial Bank of Ethiopia', 'Commercial Bank of Ethiopia'), (2, 'Bank of Abyssinia', 'Bank of Abyssinia')]


In [20]:
# Get sentiment data
def get_sentiment(rating):
    if rating >= 4:
        return 'positive'
    elif rating == 3:
        return 'neutral'
    else:
        return 'negative'

def get_sentiment_score(rating):
    return (rating - 3) / 2

df['sentiment_label'] = df['rating'].apply(get_sentiment)
df['sentiment_score'] = df['rating'].apply(get_sentiment_score)
df['identified_theme'] = 'General'

# Get bank ids
cursor.execute("SELECT bank_id, bank_name FROM banks;")
bank_map = {row[1]: row[0] for row in cursor.fetchall()}

# Insert reviews
count = 0
for _, row in df.iterrows():
    bank_id = bank_map.get(row['bank'])
    cursor.execute("""
        INSERT INTO reviews 
        (bank_id, review_text, rating, review_date, sentiment_label, sentiment_score, identified_theme, source)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
    """, (
        bank_id,
        row['review'],
        row['rating'],
        row['date'],
        row['sentiment_label'],
        row['sentiment_score'],
        row['identified_theme'],
        row['source']
    ))
    count += 1

conn.commit()
print(f"Inserted {count} reviews! ✅")

Inserted 1000 reviews! ✅


In [22]:
# Count reviews per bank
cursor.execute("""
    SELECT b.bank_name, COUNT(r.review_id) as total_reviews
    FROM banks b
    JOIN reviews r ON b.bank_id = r.bank_id
    GROUP BY b.bank_name;
""")
print("Reviews per bank:")
for row in cursor.fetchall():
    print(f"  {row[0]}: {row[1]} reviews")

# Average rating per bank
cursor.execute("""
    SELECT b.bank_name, ROUND(AVG(r.rating)::numeric, 2) as avg_rating
    FROM banks b
    JOIN reviews r ON b.bank_id = r.bank_id
    GROUP BY b.bank_name;
""")
print("\nAverage rating per bank:")
for row in cursor.fetchall():
    print(f"  {row[0]}: {row[1]} stars")

# Sentiment distribution
cursor.execute("""
    SELECT b.bank_name, r.sentiment_label, COUNT(*) as count
    FROM banks b
    JOIN reviews r ON b.bank_id = r.bank_id
    GROUP BY b.bank_name, r.sentiment_label
    ORDER BY b.bank_name;
""")
print("\nSentiment distribution:")
for row in cursor.fetchall():
    print(f"  {row[0]} - {row[1]}: {row[2]}")

Reviews per bank:
  Bank of Abyssinia: 500 reviews
  Commercial Bank of Ethiopia: 500 reviews

Average rating per bank:
  Bank of Abyssinia: 3.58 stars
  Commercial Bank of Ethiopia: 4.11 stars

Sentiment distribution:
  Bank of Abyssinia - positive: 318
  Bank of Abyssinia - neutral: 19
  Bank of Abyssinia - negative: 163
  Commercial Bank of Ethiopia - positive: 381
  Commercial Bank of Ethiopia - neutral: 30
  Commercial Bank of Ethiopia - negative: 89
